In [2]:
import os
import pandas as pd
import requests
import numpy as np
import openpyxl
import matplotlib.pyplot as plt
from dotenv import load_dotenv

In [3]:
#Visualiza la informacion completa 
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

Acceso a la API de YELP

In [4]:
# Carga las variables del archivo .env
load_dotenv()

# URL de la API
api_url = 'https://api.yelp.com/v3/businesses/search'

# credenciales de forma segura
clientid = os.getenv('YELP_CLIENT_ID')
apikey = os.getenv('YELP_API_KEY')

# Configuracion headers 
headers = {'Authorization': f'Bearer {apikey}'}

Extracción de datos para una ciudad

In [5]:
ciudad = "Denver" # Se puede elegir cualquier ciudad del dataset entregado

In [6]:
# De esta manera se consulta el API de Yelp para obtener información sobre restaurantes en la ciudad especificada
#Sólo se extraen las columnas de interes, que puden aportar valor al análisis
Complete_list = []
#Solo se permiten descargar de 50 datos hasta un max de 240 por eso se hace un ciclo
for i in range(0, 200 ,50):
    params={'term':'restaurants','location': ciudad,'limit':50, 'offset':i }
    response=requests.get(api_url,params=params,headers=headers)
    data=response.json()
    
    for info in data['businesses']:
        buffer = []
        buffer.append(info.get('id', None))
        buffer.append(info.get('name', None))
        buffer.append(info.get('is_closed',None))
        buffer.append(info.get('review_count',None))
        
        #Desempaquetar las que tiene los diccionarios dentro
        categories = info.get('categories', [])#'categories'
        if categories and isinstance(categories, list):
            principal_categories = categories[0].get('title',None)
        else:
            principal_categories = None
        buffer.append(principal_categories)
        
        buffer.append(info.get('rating',None))
        
        coordinates = info.get('coordinates')
        buffer.append(coordinates.get('latitude',None))
        buffer.append(coordinates.get('longitude',None))
        
        #Transacciones tambien se desenglozan 3 posibles opciones delivery, pick up y reservation
        trans = info.get('transactions',[])
        #Para delivery
        buffer.append('yes' if 'delivery' in trans else 'No')
        #Para reservasión
        buffer.append('yes' if 'restaurant_reservation' in trans else 'No')
        #Para 
        buffer.append('yes' if 'pickup' in trans else 'No')
       
        
        #equivalencia para los price
        
        buffer.append(info.get('price',None))

        site = info.get('location',{})
        buffer.append(site.get('city'))

        buffer.append(info.get('phone', None))
        buffer.append(info.get('display_phone', None))
        
        Complete_list.append(buffer)

    
print(Complete_list)

[['GmniFxffmpFcbmb5KWIwbA', 'Culinary Dropout', False, 1038, 'New American', 4.2, 39.73132, -104.93918, 'No', 'No', 'No', '$$', 'Denver', '+17207790190', '(720) 779-0190'], ['Y-c9rFdGYXlT-A1ndoCTrg', 'Tavernetta', False, 990, 'Italian', 4.5, 39.75302254234153, -105.00196807402817, 'yes', 'No', 'yes', '$$$', 'Denver', '+17206051889', '(720) 605-1889'], ['UkXP_D2vih-LS5oAaMxRIg', 'Madeline', False, 20, 'Restaurants', 4.9, 39.725727, -104.97143, 'No', 'No', 'No', None, 'Denver', '', ''], ['lyit8GGKG4AyxO3CrTQErQ', 'Jack’s Modern Steakhouse', False, 252, 'Bars', 4.6, 39.68985, -104.980581, 'No', 'No', 'No', '$$$', 'Denver', '+17208685800', '(720) 868-5800'], ['2wrS0h8iXFZTr4p0egCLjQ', 'West Saloon & Kitchen ', False, 1098, 'American', 4.5, 39.74403, -104.99044, 'yes', 'yes', 'yes', '$$', 'Denver', '+13038253690', '(303) 825-3690'], ['svJWwW0ilssyqk_UML0mUg', 'Work & Class', False, 1893, 'New American', 4.5, 39.75761, -104.98604, 'yes', 'No', 'No', '$$', 'Denver', '+13032920700', '(303) 292

In [7]:
#Veamos los keys del diccionario recibido
data.keys()

dict_keys(['businesses', 'total', 'region'])

In [8]:
#El segundo elemento del diccionario indica el total de restaurants existentes en la API
print('En total la base de datos registra %d restaurants'%data['total'])

En total la base de datos registra 4100 restaurants


Creacion del DataFrame

In [9]:
# Se crea en dataframe con la info seleccionada
Columns= ['id','name_business','is_closed','review_count','categories','rating','latitude', 'longitude','delivery', 'restaurant_reservation','pickup','price','city','phone','display_phone']
dt_restaurant_YELP= pd.DataFrame(Complete_list, columns=Columns)
print(dt_restaurant_YELP)

                         id                           name_business  \
0    GmniFxffmpFcbmb5KWIwbA                        Culinary Dropout   
1    Y-c9rFdGYXlT-A1ndoCTrg                              Tavernetta   
2    UkXP_D2vih-LS5oAaMxRIg                                Madeline   
3    lyit8GGKG4AyxO3CrTQErQ                Jack’s Modern Steakhouse   
4    2wrS0h8iXFZTr4p0egCLjQ                  West Saloon & Kitchen    
5    svJWwW0ilssyqk_UML0mUg                            Work & Class   
6    0jTb4q8Ezil_tXkbZf-HRQ                                Rougarou   
7    ywFvkNZVJoXUQbZoq7fr1w                                   Rioja   
8    i8VF3q7dTszCcigJSMcJzQ                        Angelo’s Taverna   
9    iS9j0iSOhRzBBYJdzzuZFQ                              Point Easy   
10   jML03gtMkgIFWX9Iium4Xg                                   Safta   
11   x0rVT3E__00xg99uGr0tjg                            Happy Camper   
12   InKe17J_fpqDBorFxEsUXw                  Ma's Kitchen Next Door   
13   g

Verificación de datos

In [10]:
#Revisión de información general del dataFrame
dt_restaurant_YELP.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      200 non-null    str    
 1   name_business           200 non-null    str    
 2   is_closed               200 non-null    bool   
 3   review_count            200 non-null    int64  
 4   categories              200 non-null    str    
 5   rating                  200 non-null    float64
 6   latitude                200 non-null    float64
 7   longitude               200 non-null    float64
 8   delivery                200 non-null    str    
 9   restaurant_reservation  200 non-null    str    
 10  pickup                  200 non-null    str    
 11  price                   127 non-null    str    
 12  city                    200 non-null    str    
 13  phone                   200 non-null    str    
 14  display_phone           200 non-null    str    
dtype

Eliminacion de duplicados

In [11]:
#Se verifica locales duplicados a través del id y se verifica con Data Wrangler que sean unicos verificando que los valores de latitud y longitus sean diferentes
Duplicated_Rows=dt_restaurant_YELP[dt_restaurant_YELP.duplicated(subset=['id'])]
print(Duplicated_Rows)

Empty DataFrame
Columns: [id, name_business, is_closed, review_count, categories, rating, latitude, longitude, delivery, restaurant_reservation, pickup, price, city, phone, display_phone]
Index: []


Limpieza de datos

In [12]:
#Revision general de posibles datos atípicos
dt_restaurant_YELP.describe()
#No se encuentran anomalías

,review_count,rating,latitude,longitude
count,200.000000,200.000000,200.000000,200.000000
mean,603.385000,4.338500,39.743100,-104.980427
std,813.444365,0.310726,0.021372,0.024719
min,1.000000,3.400000,39.676981,-105.063120
25%,93.250000,4.100000,39.735205,-104.996371
50%,331.500000,4.400000,39.745705,-104.982462
75%,766.750000,4.500000,39.758118,-104.969122
max,4998.000000,5.000000,39.778458,-104.870020


Estandarización de texto str

In [13]:
# Se seleccionan las columnas tipo Str
Columns_Text = dt_restaurant_YELP.select_dtypes(include=[str]).columns
Col_tex=Columns_Text.drop(['id','price','phone','display_phone'])
print(Col_tex)

#Se normalizan las columnas tipo str todas a minusculas y con un soloo
for col in Col_tex:
    dt_restaurant_YELP[col]=dt_restaurant_YELP[col].astype(str).str.strip().str.lower()
    print(dt_restaurant_YELP)

Index(['name_business', 'categories', 'delivery', 'restaurant_reservation',
       'pickup', 'city'],
      dtype='str')


                         id                           name_business  \
0    GmniFxffmpFcbmb5KWIwbA                        culinary dropout   
1    Y-c9rFdGYXlT-A1ndoCTrg                              tavernetta   
2    UkXP_D2vih-LS5oAaMxRIg                                madeline   
3    lyit8GGKG4AyxO3CrTQErQ                jack’s modern steakhouse   
4    2wrS0h8iXFZTr4p0egCLjQ                   west saloon & kitchen   
5    svJWwW0ilssyqk_UML0mUg                            work & class   
6    0jTb4q8Ezil_tXkbZf-HRQ                                rougarou   
7    ywFvkNZVJoXUQbZoq7fr1w                                   rioja   
8    i8VF3q7dTszCcigJSMcJzQ                        angelo’s taverna   
9    iS9j0iSOhRzBBYJdzzuZFQ                              point easy   
10   jML03gtMkgIFWX9Iium4Xg                                   safta   
11   x0rVT3E__00xg99uGr0tjg                            happy camper   
12   InKe17J_fpqDBorFxEsUXw                  ma's kitchen next door   
13   g

Eliminar ciudades diferentes a la seleccionada

In [14]:
#Eliminación de información no pprteneciente a Denver
total_no_denver = (dt_restaurant_YELP['city'] != 'denver').sum()
print(total_no_denver)

# Se sobreescribe el dataframe con valores pertencientes solo a Denver
dt_restaurant_YELP= dt_restaurant_YELP[dt_restaurant_YELP['city']== 'denver']
print(dt_restaurant_YELP)

4
                         id                           name_business  \
0    GmniFxffmpFcbmb5KWIwbA                        culinary dropout   
1    Y-c9rFdGYXlT-A1ndoCTrg                              tavernetta   
2    UkXP_D2vih-LS5oAaMxRIg                                madeline   
3    lyit8GGKG4AyxO3CrTQErQ                jack’s modern steakhouse   
4    2wrS0h8iXFZTr4p0egCLjQ                   west saloon & kitchen   
5    svJWwW0ilssyqk_UML0mUg                            work & class   
6    0jTb4q8Ezil_tXkbZf-HRQ                                rougarou   
7    ywFvkNZVJoXUQbZoq7fr1w                                   rioja   
8    i8VF3q7dTszCcigJSMcJzQ                        angelo’s taverna   
9    iS9j0iSOhRzBBYJdzzuZFQ                              point easy   
10   jML03gtMkgIFWX9Iium4Xg                                   safta   
11   x0rVT3E__00xg99uGr0tjg                            happy camper   
12   InKe17J_fpqDBorFxEsUXw                  ma's kitchen next door   
13  

In [15]:
dt_restaurant_YELP['phone'] = dt_restaurant_YELP['phone'].replace('', np.nan)
dt_restaurant_YELP['display_phone'] = dt_restaurant_YELP['display_phone'].replace('', np.nan)

Datos nulos

In [16]:
Percentage_Null = dt_restaurant_YELP.isna().mean()*100
print(Percentage_Null)

id                         0.000000
name_business              0.000000
is_closed                  0.000000
review_count               0.000000
categories                 0.000000
rating                     0.000000
latitude                   0.000000
longitude                  0.000000
delivery                   0.000000
restaurant_reservation     0.000000
pickup                     0.000000
price                     36.734694
city                       0.000000
phone                      6.122449
display_phone              6.122449
dtype: float64


In [17]:
#Price se llena con Desconocido, porque queremos al estar escrito en tipo simbolo, evitamos modificar la tendencia
#Ya que faltan un 37% de los datos y usar la moda haría que incurriera en un sesgo grande
dt_restaurant_YELP['price']=dt_restaurant_YELP['price'].fillna('Desconocido')

In [18]:
#Se rellena los valores nulos con 'desconicido' para no inventar informacion falsa
dt_restaurant_YELP['phone']=dt_restaurant_YELP['phone'].fillna('Desconocido')

In [19]:
#Se rellena los valores nulos con 'desconicido' para no inventar informacion falsa
dt_restaurant_YELP['display_phone']=dt_restaurant_YELP['display_phone'].fillna('Desconocido')

In [20]:
Percentage_Null = dt_restaurant_YELP.isna().mean()*100
print(Percentage_Null)

id                        0.0
name_business             0.0
is_closed                 0.0
review_count              0.0
categories                0.0
rating                    0.0
latitude                  0.0
longitude                 0.0
delivery                  0.0
restaurant_reservation    0.0
pickup                    0.0
price                     0.0
city                      0.0
phone                     0.0
display_phone             0.0
dtype: float64


In [21]:
dt_restaurant_YELP

,id,name_business,is_closed,review_count,categories,rating,latitude,longitude,delivery,restaurant_reservation,pickup,price,city,phone,display_phone
0,GmniFxffmpFcbmb5KWIwbA,culinary dropout,False,1038,new american,4.2,39.731320,-104.939180,no,no,no,$$,denver,+17207790190,(720) 779-0190
1,Y-c9rFdGYXlT-A1ndoCTrg,tavernetta,False,990,italian,4.5,39.753023,-105.001968,yes,no,yes,$$$,denver,+17206051889,(720) 605-1889
2,UkXP_D2vih-LS5oAaMxRIg,madeline,False,20,restaurants,4.9,39.725727,-104.971430,no,no,no,Desconocido,denver,Desconocido,Desconocido
3,lyit8GGKG4AyxO3CrTQErQ,jack’s modern steakhouse,False,252,bars,4.6,39.689850,-104.980581,no,no,no,$$$,denver,+17208685800,(720) 868-5800
4,2wrS0h8iXFZTr4p0egCLjQ,west saloon & kitchen,False,1098,american,4.5,39.744030,-104.990440,yes,yes,yes,$$,denver,+13038253690,(303) 825-3690
5,svJWwW0ilssyqk_UML0mUg,work & class,False,1893,new american,4.5,39.757610,-104.986040,yes,no,no,$$,denver,+13032920700,(303) 292-0700
6,0jTb4q8Ezil_tXkbZf-HRQ,rougarou,False,59,southern,4.7,39.756240,-104.975920,no,no,no,Desconocido,denver,Desconocido,Desconocido
7,ywFvkNZVJoXUQbZoq7fr1w,rioja,False,2930,new american,4.3,39.747808,-104.999331,no,no,yes,$$$,denver,+13038202282,(303) 820-2282
8,i8VF3q7dTszCcigJSMcJzQ,angelo’s taverna,False,2033,italian,4.4,39.725523,-104.979224,yes,no,yes,$$,denver,+13037443366,(303) 744-3366
9,iS9j0iSOhRzBBYJdzzuZFQ,point easy,False,94,new american,4.6,39.756880,-104.963220,no,no,no,Desconocido,denver,+13032335656,(303) 233-5656


In [22]:
#Reservation
Dic_res=dict(Cantidad = dt_restaurant_YELP['restaurant_reservation'].value_counts(), Porcentaje = dt_restaurant_YELP['restaurant_reservation'].value_counts(normalize = True)*100)
Res_Table = pd.DataFrame(Dic_res)
Res_Table= Res_Table.reindex(['no','yes'])
Res_Table.loc['Total']=Res_Table.sum()
print(Res_Table)
#Delivery
Dic_del=dict(Cantidad = dt_restaurant_YELP['delivery'].value_counts(), Porcentaje = dt_restaurant_YELP['delivery'].value_counts(normalize = True)*100)
Del_Table = pd.DataFrame(Dic_del)
Del_Table= Del_Table.reindex(['no','yes'])
Del_Table.loc['Total']=Del_Table.sum()
print(Del_Table)
#Pickup
Dic_pic=dict(Cantidad = dt_restaurant_YELP['pickup'].value_counts(), Porcentaje = dt_restaurant_YELP['pickup'].value_counts(normalize = True)*100)
Pic_Table = pd.DataFrame(Dic_pic)
Pic_Table= Pic_Table.reindex(['no','yes'])
Pic_Table.loc['Total']=Pic_Table.sum()
print(Pic_Table)

                        Cantidad  Porcentaje
restaurant_reservation                      
no                         190.0   96.938776
yes                          6.0    3.061224
Total                      196.0  100.000000
          Cantidad  Porcentaje
delivery                      
no            85.0   43.367347
yes          111.0   56.632653
Total        196.0  100.000000
        Cantidad  Porcentaje
pickup                      
no         103.0    52.55102
yes         93.0    47.44898
Total      196.0   100.00000
